# Comparación de Pipelines de Registro Histológico: Sift_Demons (SIFT+ECC+Demons) vs DeeperHistReg vs VALIS

**Objetivo:** comparar de forma visual y cuantitativa los tiles pareados producidos por 3 pipelines de
registro distintos, sobre **dos muestras** (Muestra1 y Muestra3), para justificar **cuál pipeline produce una alineación de mayor calidad**.

## Estructura de datos esperada 

| Dataset de Kaggle | Contenido | Rol |
|---|---|---|
| `Muestra1Nefro` | `muestra_1.tif`, `muestra_2.tif`, `offsets_recorte.json` | TIFs originales |
| `Muestra3Nefro` | ídem | TIFFs originales |
| `TilesSift_DemonsMuestra1` | `ecc/`, `geom/`, `manifest.csv` | Tiles del pipeline **Sift_Demons (SIFT+ECC+Demons)**, Muestra 1 |
| `TilesSift_DemonsMuestra3` | ídem | Pipeline Sift_Demons, Muestra 3 |
| `TilesDeeperhistregMuestra1` | `ecc/`, `geom/`, `indice_tiles.csv` | Tiles del pipeline **DeeperHistReg**, Muestra 1 |
| `TilesDeeperhistregMuestra3` | ídem | Pipeline DeeperHistReg, Muestra 3 |
| `TilesValisMuestra1` | `ecc/`, `geom/`, `manifest.csv` | Tiles del pipeline **VALIS**, Muestra 1 |
| `TilesValisMuestra3` | ídem | Pipeline VALIS, Muestra 3 |

## Set de métricas usado en el paper (escenario monomodal: Masson vs Masson)

Como Tejido 1 y Tejido 2 comparten la **misma tinción** (tricrómico de Masson), no hace falta compensar
diferencias cromáticas entre tinciones distintas. El paper fija, como resultado de ese análisis, **cinco
métricas principales, sin referencia**, que cubren distancia física, correspondencia de intensidad/textura,
superposición geométrica y plausibilidad física de la deformación:

| # | Métrica | Qué mide | Rol en el paper |
|---|---|---|---|
| 1 | **PCC-TRE** | Desplazamiento residual (px) por correlación cruzada de fase | Métrica primaria de traslación |
| 2 | **NCC** | Correlación cruzada normalizada de intensidades | Métrica primaria de correspondencia (monomodal) |
| 3 | **SSIM enmascarado** | Luminancia/contraste/estructura, restringido a la región de tejido | Métrica primaria de correspondencia (reemplaza a SSIM estándar, ver más abajo) |
| 4 | **Mask IoU (Otsu)** | Superposición geométrica de las máscaras de tejido | Control independiente, invariante a la tinción |
| 5 | **Folding ratio (%)** | % de píxeles con Jacobiano ≤ 0 en el campo de deformación (proxy por flujo óptico) | Control de integridad física de la deformación, invariante a la tinción |

**Por qué SSIM enmascarado y no SSIM estándar:** el SSIM estándar calculado sobre el tile completo queda
inflado por el fondo blanco compartido (idéntico en T1 y T2, fuera del tejido), lo que sobreestima la
similitud real.

# CONFIGURACION

Esta celda busca, dentro de `/kaggle/input/`, los datasets correspondientes a cada muestra y pipeline
por coincidencia aproximada de nombre (ignorando mayúsculas, guiones y guiones bajos), y localiza el
CSV manifiesto de forma recursiva dentro de cada uno. Si tu entorno no es Kaggle o los nombres reales
de los datasets son distintos, editá el diccionario `RUTAS_MANUALES` para forzar rutas específicas.

**Emparejamiento de tiles entre pipelines (misma región de tejido, dentro de cada muestra)**

Emparejar por coordenadas absolutas (`x1_orig`, `y1_orig`), pero ahora **por
muestra**: un tile de `Muestra1` nunca se empareja con uno de `Muestra3`, aunque tengan coordenadas
parecidas, porque son slides físicamente distintas.

In [ ]:
# Configuracion
!pip install -q scikit-image scipy seaborn statsmodels

import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from skimage.registration import phase_cross_correlation
from skimage.metrics import structural_similarity as ssim
from skimage.filters import threshold_otsu
from sklearn.metrics import normalized_mutual_info_score
from scipy.stats import wilcoxon
from scipy.spatial import cKDTree

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 50)
np.random.seed(42)

TILE_SIZE = 512
TOLERANCIA_MATCH_PX = TILE_SIZE // 2

CARPETA_SALIDA = "/kaggle/working/comparativa_pipelines"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

print(f"Pipelines:")
PIPELINES = ["Sift_Demons", "DeeperHistReg", "VALIS"]
MUESTRAS = ["Muestra1", "Muestra3"]

# Ruta al manifiesto de cada pipeline
RUTAS_MANIFIESTOS = {
    ("Muestra1", "Sift_Demons"):   "/kaggle/input/datasets/candelapaez/tilessift-demonsmuestra1/manifest.csv",
    ("Muestra1", "DeeperHistReg"): "/kaggle/input/datasets/candelapaez/tilesdeeperhistregmuestra1/indice_tiles.csv",
    ("Muestra1", "VALIS"):         "/kaggle/input/datasets/tobiasdelgado/107514-40x-valis/107514-40x_tiles_tile512_nivel0_valis/indice_tiles.csv",
    ("Muestra3", "Sift_Demons"):   "/kaggle/input/datasets/candelapaez/tilessift-demonsmuestra3/manifest.csv",
    ("Muestra3", "DeeperHistReg"): "/kaggle/input/datasets/candelapaez/tilesdeeperhistregmuestra3/indice_tiles.csv",
    ("Muestra3", "VALIS"):         "/kaggle/input/datasets/tobiasdelgado/20260518-182923scanslide-valis/20260518-182923ScanSlide_tiles_tile512_nivel0_valis/indice_tiles.csv",
}

COLUMNAS_MANIFIESTO = [
    "pipeline", "muestra", "nombre", "x1", "y1", "x1_orig", "y1_orig", "ruta_t1", "ruta_t2",
    "ecc_exitoso", "informatividad_t1", "informatividad_t2",
    "shift_residual_px", "confianza_alineacion",
]


def cargar_manifiesto(manifest_path, id_muestra, pipeline_name):
    """Lee un manifest.csv/indice_tiles.csv y SIEMPRE reconstruye ruta_t1/ruta_t2
    a partir de base_dir + nombre — ignora cualquier ruta_t1/ruta_t2 que ya
    traiga el CSV, porque esas quedaron grabadas del entorno donde se generó
    el dataset originalmente (p.ej. /kaggle/working/...) y ya no son válidas acá."""
    base_dir = os.path.dirname(manifest_path)
    df = pd.read_csv(manifest_path)

    if pipeline_name == "VALIS":
        df["ruta_t1"] = [os.path.join(base_dir, "muestra_1", n) for n in df["nombre"]]
        df["ruta_t2"] = [os.path.join(base_dir, "muestra_2", n) for n in df["nombre"]]
    else:
        if "carpeta" in df.columns:
            carpetas = df["carpeta"]
        elif "ecc_exitoso" in df.columns:
            carpetas = df["ecc_exitoso"].map(lambda ok: "ecc" if ok else "geom")
        else:
            carpetas = ["ecc"] * len(df)
        df["ruta_t1"] = [os.path.join(base_dir, c, "tejido1", n) for c, n in zip(carpetas, df["nombre"])]
        df["ruta_t2"] = [os.path.join(base_dir, c, "tejido2", n) for c, n in zip(carpetas, df["nombre"])]

    df["pipeline"] = pipeline_name
    df["muestra"] = id_muestra
    cols = [c for c in COLUMNAS_MANIFIESTO if c in df.columns]
    return df[cols].copy()


# Carga de los 6 manifiestos
listas_por_pipeline = {pip: [] for pip in PIPELINES}
for (id_m, pipeline_name), manifest_path in RUTAS_MANIFIESTOS.items():
    assert os.path.exists(manifest_path), f"No existe el manifiesto: [{id_m}/{pipeline_name}] {manifest_path}"
    listas_por_pipeline[pipeline_name].append(cargar_manifiesto(manifest_path, id_m, pipeline_name))

dfs_por_pipeline = {pip: pd.concat(listas, ignore_index=True) for pip, listas in listas_por_pipeline.items()}

# Verificación de que los archivos existan
for pipeline_name, df in dfs_por_pipeline.items():
    existe = [os.path.exists(t1) and os.path.exists(t2) for t1, t2 in zip(df["ruta_t1"], df["ruta_t2"])]
    df["existe"] = existe
    conteo_por_muestra = df.groupby("muestra")["existe"].sum().to_dict()
    print(f"{pipeline_name}: {len(df)} tiles en manifiesto(s),{conteo_por_muestra}")
          # f"(por muestra: {conteo_por_muestra})")
    dfs_por_pipeline[pipeline_name] = df[df["existe"]].drop(columns="existe").reset_index(drop=True)

# Nombres de variable preservados para no romper el resto del notebook
df_sift_demons = dfs_por_pipeline["Sift_Demons"]
df_dhr = dfs_por_pipeline["DeeperHistReg"]
df_valis = dfs_por_pipeline["VALIS"]


# Emparejamiento por coordenadas absolutas (x1_orig, y1_orig), por muestra
def emparejar_multipipeline(dfs_por_pipeline, tol_px=TOLERANCIA_MATCH_PX):
    """Empareja tiles de N pipelines por coordenadas absolutas, usando el primero
    del diccionario como referencia y buscando el vecino más cercano en el resto."""
    nombres_pipeline = list(dfs_por_pipeline.keys())
    df_ref = dfs_por_pipeline[nombres_pipeline[0]]
    otros = nombres_pipeline[1:]

    if len(df_ref) == 0 or any(len(dfs_por_pipeline[nom]) == 0 for nom in otros):
        return pd.DataFrame()

    coords_ref = df_ref[["x1_orig", "y1_orig"]].to_numpy(dtype=float)
    arboles = {nom: (cKDTree(dfs_por_pipeline[nom][["x1_orig", "y1_orig"]].to_numpy(dtype=float)), dfs_por_pipeline[nom])
               for nom in otros}
    usados = {nom: set() for nom in otros}

    filas = []
    for i_ref in range(len(df_ref)):
        fila_ref = df_ref.iloc[i_ref]
        fila_dict = {
            "muestra": fila_ref["muestra"],
            "x1_orig": fila_ref["x1_orig"], "y1_orig": fila_ref["y1_orig"],
            f"nombre_{nombres_pipeline[0]}": fila_ref["nombre"],
            f"ruta_t1_{nombres_pipeline[0]}": fila_ref["ruta_t1"], f"ruta_t2_{nombres_pipeline[0]}": fila_ref["ruta_t2"],
        }
        ok = True
        dist_max = 0.0
        for nom in otros:
            arbol, df_o = arboles[nom]
            dist, idx = arbol.query(coords_ref[i_ref], k=1)
            if dist > tol_px or idx in usados[nom]:
                ok = False
                break
            usados[nom].add(idx)
            dist_max = max(dist_max, dist)
            fila_o = df_o.iloc[idx]
            fila_dict[f"nombre_{nom}"] = fila_o["nombre"]
            fila_dict[f"ruta_t1_{nom}"] = fila_o["ruta_t1"]
            fila_dict[f"ruta_t2_{nom}"] = fila_o["ruta_t2"]
        fila_dict["distancia_match_px"] = dist_max
        if ok:
            filas.append(fila_dict)
    return pd.DataFrame(filas)


piezas_pares = []

for id_m in MUESTRAS:
    dfs_m = {
        "Sift_Demons": df_sift_demons[df_sift_demons["muestra"] == id_m],
        "DeeperHistReg": df_dhr[df_dhr["muestra"] == id_m],
        "VALIS": df_valis[df_valis["muestra"] == id_m],
    }
    if any(len(dfs_m[nom]) == 0 for nom in PIPELINES):
        print(f"[{id_m}] falta alguno de los tres pipelines, no se puede emparejar.")
        continue
    pares_m = emparejar_multipipeline(dfs_m, tol_px=TOLERANCIA_MATCH_PX)
    print(f"\n[{id_m}] tiles emparejados: {len(pares_m)}")
    piezas_pares.append(pares_m)

df_pares_comunes = pd.concat(piezas_pares, ignore_index=True) if piezas_pares else pd.DataFrame()
print(f"\nTotal tiles emparejados (ambas muestras): {len(df_pares_comunes)}")
if len(df_pares_comunes) > 0:
    print(f"\nDistancia de emparejamiento -> mediana: {df_pares_comunes['distancia_match_px'].median():.1f}px "
          f"| máx: {df_pares_comunes['distancia_match_px'].max():.1f}px (tolerancia usada: {TOLERANCIA_MATCH_PX}px)")

# Comparación visual lado a lado

Muestra tiles de la misma región de tejido procesados por ambos pipelines. Se toma una muestra aleatoria mezclando ambas muestras (Muestra1 y Muestra3) salvo que se fije `solo_muestra`.


In [ ]:
# VISUALIZACIÓN LADO A LADO (misma región, 3 pipelines)
def cargar_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def comparar_visualmente(df_pares, n_muestras=5, seed=42, solo_muestra=None):
    df_sel = df_pares if solo_muestra is None else df_pares[df_pares["muestra"] == solo_muestra]
    if len(df_sel) == 0:
        print("No hay tiles comunes para comparar. Revisá TOLERANCIA_MATCH_PX o las rutas de los manifiestos.")
        return

    muestra = df_sel.sample(min(n_muestras, len(df_sel)), random_state=seed).to_dict("records")

    n_cols = 3 * len(PIPELINES)
    fig, axes = plt.subplots(len(muestra), n_cols, figsize=(4.6 * n_cols, 4.6 * len(muestra)))
    if len(muestra) == 1:
        axes = [axes]

    for i, par in enumerate(muestra):
        titulos, imagenes = [], []
        for pip in PIPELINES:
            t1 = cargar_rgb(par[f"ruta_t1_{pip}"])
            t2 = cargar_rgb(par[f"ruta_t2_{pip}"])
            overlay = cv2.addWeighted(t1, 0.5, t2, 0.5, 0)
            titulos += ["T1 Masson (ref)", f"T2 {pip}", f"Overlay {pip}"]
            imagenes += [t1, t2, overlay]

        for j in range(n_cols):
            axes[i][j].imshow(imagenes[j])
            axes[i][j].set_title(titulos[j], fontsize=11)
            axes[i][j].axis("off")

        axes[i][0].text(-0.15, 0.5, f"{par['muestra']}\n({par['x1_orig']:.0f},{par['y1_orig']:.0f})",
                         transform=axes[i][0].transAxes, rotation=90,
                         va="center", ha="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_SALIDA, "comparacion_visual.png"), dpi=150, bbox_inches="tight")
    plt.show()


comparar_visualmente(df_pares_comunes, n_muestras=10)

# Métricas

Se calcula, por tile, el conjunto completo de métricas sin referencia, sobre **todos** los tiles válidos de cada pipeline (ambas muestras combinadas), etiquetando cada fila con `muestra` y `pipeline` para poder desagregar después.

In [ ]:
# MÉTRICAS
import sys
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from skimage.registration import phase_cross_correlation
from skimage.metrics import structural_similarity as ssim
from skimage.filters import threshold_otsu


def to_gray(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)


# PCC-TRE
def calc_pcc_tre(fixed_rgb, moving_rgb):
    fixed = to_gray(fixed_rgb).astype(np.float32)
    moving = to_gray(moving_rgb).astype(np.float32)
    shift, error, diffphase = phase_cross_correlation(fixed, moving, upsample_factor=10)
    tre = float(np.linalg.norm(shift))
    return tre, shift


# NCC 
def calc_ncc(fixed_rgb, moving_rgb):
    fixed = to_gray(fixed_rgb).astype(np.float64)
    moving = to_gray(moving_rgb).astype(np.float64)
    f_mean = fixed - np.mean(fixed)
    m_mean = moving - np.mean(moving)
    num = np.sum(f_mean * m_mean)
    den = np.sqrt(np.sum(f_mean ** 2) * np.sum(m_mean ** 2))
    return float(num / den) if den != 0.0 else 0.0


# SSIM enmascarado
def calc_ssim_enmascarado(fixed_rgb, moving_rgb, umbral_blanco=230):
    fixed_gray = to_gray(fixed_rgb)
    moving_gray = to_gray(moving_rgb)
    _, mapa_ssim = ssim(fixed_gray, moving_gray, data_range=255, full=True)
    mascara_tejido = (fixed_gray < umbral_blanco) | (moving_gray < umbral_blanco)
    if np.sum(mascara_tejido) == 0:
        return 0.0
    return float(np.mean(mapa_ssim[mascara_tejido]))


# Mask IoU (Otsu)
def calc_mask_iou(fixed_rgb, moving_rgb):
    fixed = to_gray(fixed_rgb)
    moving = to_gray(moving_rgb)
    try:
        mask_f = fixed < threshold_otsu(fixed)
        mask_m = moving < threshold_otsu(moving)
    except ValueError:
        return 0.0
    inter = np.logical_and(mask_f, mask_m)
    union = np.logical_or(mask_f, mask_m)
    return float(np.sum(inter) / np.sum(union)) if np.sum(union) > 0 else 0.0


def clasificar_iou(iou):
    if iou >= 0.80: return "Excelente"
    if iou >= 0.70: return "Bueno"
    if iou >= 0.64: return "Aceptable"
    return "Fallo"


# Jacobiano / NJD 
def calc_jacobian_proxy(fixed_rgb, moving_rgb):
    fixed = to_gray(fixed_rgb)
    moving = to_gray(moving_rgb)
    flow = cv2.calcOpticalFlowFarneback(
        fixed, moving, None,
        pyr_scale=0.5, levels=3, winsize=21,
        iterations=5, poly_n=5, poly_sigma=1.2, flags=0
    )
    disp_x, disp_y = flow[..., 0], flow[..., 1]
    du_x_dy, du_x_dx = np.gradient(disp_x)
    du_y_dy, du_y_dx = np.gradient(disp_y)
    jacobian = (1.0 + du_x_dx) * (1.0 + du_y_dy) - du_x_dy * du_y_dx
    folding_ratio_pct = float(np.mean(jacobian <= 0.0) * 100.0)
    return jacobian, folding_ratio_pct


UMBRAL_TISSUE_RATIO = 0.10  # UWarp/CurvReg: descartar tiles con <10% de tejido activo

def calc_tissue_ratio_gray(gray_img):
    try:
        umbral = threshold_otsu(gray_img)
    except ValueError:
        return 0.0
    mascara_tejido = gray_img < umbral
    return float(np.sum(mascara_tejido) / mascara_tejido.size)


print("Métricas definidas:")
print("  PRINCIPALES (monomodal): PCC-TRE, NCC, SSIM enmascarado, Mask IoU, Jacobiano/NJD")


def _procesar_fila(row, nombre_pipeline):
    cv2.setNumThreads(1)
    try:
        t1 = cargar_rgb(row["ruta_t1"])
        t2 = cargar_rgb(row["ruta_t2"])
        t1_gray = to_gray(t1)
        t2_gray = to_gray(t2)

        tre, _ = calc_pcc_tre(t1, t2)
        ncc = calc_ncc(t1, t2)
        ssim_enmascarado = calc_ssim_enmascarado(t1, t2)
        iou = calc_mask_iou(t1, t2)
        _, folding_pct = calc_jacobian_proxy(t1, t2)
        tissue_ratio_t1 = calc_tissue_ratio_gray(t1_gray)
        tissue_ratio_t2 = calc_tissue_ratio_gray(t2_gray)
        tre_confiable = bool(tissue_ratio_t1 >= UMBRAL_TISSUE_RATIO and tissue_ratio_t2 >= UMBRAL_TISSUE_RATIO)

        return {
            "pipeline": nombre_pipeline,
            "muestra": row.get("muestra", "NA"),
            "nombre": row["nombre"],
            "x1_orig": row.get("x1_orig", np.nan),
            "y1_orig": row.get("y1_orig", np.nan),
            "PCC_TRE_px": tre,
            "TRE_confiable": tre_confiable,
            "Tissue_ratio_T1": tissue_ratio_t1,
            "Tissue_ratio_T2": tissue_ratio_t2,
            "NCC": ncc,
            "SSIM_enmascarado": ssim_enmascarado,
            "Mask_IoU": iou,
            "IoU_categoria": clasificar_iou(iou),
            "Folding_ratio_pct": folding_pct,
            "confianza_alineacion_interna": row.get("confianza_alineacion", np.nan),
            "_error": None,
        }
    except Exception as e:
        return {"pipeline": nombre_pipeline, "nombre": row.get("nombre", "NA"), "_error": str(e)}


N_JOBS = max(1, (os.cpu_count() or 4) - 1)


def calcular_metricas_dataframe(df, nombre_pipeline, limite=None, n_jobs=N_JOBS):
    if len(df) == 0:
        print(f"[{nombre_pipeline}] dataframe vacío, se omite.")
        return pd.DataFrame()

    subset = df if limite is None else df.sample(min(limite, len(df)), random_state=42)
    filas = subset.to_dict("records")

    print(f"Iniciando {nombre_pipeline}: {len(filas)} tiles, {n_jobs} procesos", flush=True)
    resultados = []
    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        futuros = [executor.submit(_procesar_fila, fila, nombre_pipeline) for fila in filas]
        for futuro in tqdm(as_completed(futuros), total=len(futuros),
                            desc=f"Métricas {nombre_pipeline}", file=sys.stdout):
            resultados.append(futuro.result())
    print(f"Terminado {nombre_pipeline}", flush=True)

    errores = [r for r in resultados if r.get("_error")]
    for e in errores:
        print(f"  [WARN] fallo en {e.get('nombre')} ({nombre_pipeline}): {e['_error']}")

    filas_ok = [{k: v for k, v in r.items() if k != "_error"} for r in resultados if not r.get("_error")]
    return pd.DataFrame(filas_ok)


LIMITE_TILES = None  

PIPELINES_A_EVALUAR = [
    ("Sift_Demons", df_sift_demons),
    ("DeeperHistReg", df_dhr),
    ("VALIS", df_valis),
]

resultados_por_pipeline = {}
for nombre_pipeline, df_pipeline in PIPELINES_A_EVALUAR:
    resultados_por_pipeline[nombre_pipeline] = calcular_metricas_dataframe(
        df_pipeline, nombre_pipeline, limite=LIMITE_TILES
    )

df_metricas_sift_demons = resultados_por_pipeline["Sift_Demons"]
df_metricas_dhr = resultados_por_pipeline["DeeperHistReg"]
df_metricas_valis = resultados_por_pipeline["VALIS"]

# Verificar si algún pipeline quedó vacío.
for nombre_pipeline, df_res in resultados_por_pipeline.items():
    assert len(df_res) > 0, f"[{nombre_pipeline}] no generó ninguna métrica — revisar manifiesto/rutas."

df_metricas_todas = pd.concat(
    list(resultados_por_pipeline.values()), ignore_index=True
).reset_index(drop=True)
df_metricas_todas.to_csv(os.path.join(CARPETA_SALIDA, "metricas_por_tile.csv"), index=False)

print(f"\nTotal tiles evaluados -> Sift_Demons: {len(df_metricas_sift_demons)} | "
      f"DeeperHistReg: {len(df_metricas_dhr)} | VALIS: {len(df_metricas_valis)}")

print("\nTiles por muestra y pipeline:")
print(df_metricas_todas.groupby(["muestra", "pipeline"]).size())

print(f"\nTiles con TRE=0.0 potencialmente no confiables (bajo contraste): "
      f"{(~df_metricas_todas['TRE_confiable']).sum()} de {len(df_metricas_todas)}")

print("\nMuestra representativa (2 tiles por pipeline):")
display(df_metricas_todas.groupby("pipeline", group_keys=False).head(2))

# INSPECCIÓN DE TILE POR COORDENADAS
Con `encontrar_tile_mas_cercano` busca, en `df_sift_demons`, `df_dhr` y `df_valis` (los manifiestos, que
solo tienen rutas y coordenadas, no métricas), cuál es el tile más cercano a (12288, 27024) en Muestra1.
Eso te da el nombre exacto del archivo de cada pipeline (ej. `tile_00123_r280_c2304.png`).

Con esos 3 nombres, esta línea filtra el dataframe grande que ya tenías (`df_metricas_todas`, generado
por el bloque de `ProcessPoolExecutor` de antes) y arma la tabla comparativa de la región, mostrando
** las 5 métricas principales del paper** (PCC-TRE, NCC, SSIM enmascarado, Mask IoU, Folding
ratio) — las mismas que arman las Tablas 3 y 4 del paper (Figuras 2 y 3). 

In [ ]:
# 3 PARES DE TILES COMPARADOS 
METRICAS_PRINCIPALES = ["PCC_TRE_px", "NCC", "SSIM_enmascarado", "Mask_IoU", "Folding_ratio_pct"]
PIPELINES_DFS = {"Sift_Demons": df_sift_demons, "DeeperHistReg": df_dhr, "VALIS": df_valis}

REGISTRO_REGIONES = []


def encontrar_tile_mas_cercano(df_pipeline, id_muestra, x_obj, y_obj, tol_px=TOLERANCIA_MATCH_PX):
    """Busca, dentro de un pipeline y una muestra, el tile más cercano a (x_obj, y_obj).
    Devuelve None si no hay ningún tile dentro de la tolerancia."""
    sub = df_pipeline[df_pipeline["muestra"] == id_muestra]
    if sub.empty:
        return None
    dist = np.hypot(sub["x1_orig"] - x_obj, sub["y1_orig"] - y_obj)
    idx_min = dist.idxmin()
    if dist.loc[idx_min] > tol_px:
        return None
    fila = sub.loc[idx_min].copy()
    fila["distancia_al_objetivo_px"] = dist.loc[idx_min]
    return fila


def _buscar_tiles_en_region(id_muestra, x_obj, y_obj, tol_px):
    """Busca el tile más cercano en cada pipeline; avisa si alguno no tiene match."""
    tiles = {}
    for nombre_pipeline, df_p in PIPELINES_DFS.items():
        fila = encontrar_tile_mas_cercano(df_p, id_muestra, x_obj, y_obj, tol_px)
        if fila is None:
            print(f"[{nombre_pipeline}] sin tile cerca de ({x_obj:.0f},{y_obj:.0f}) en {id_muestra}")
        else:
            tiles[nombre_pipeline] = fila
    return tiles


def _graficar_overlay_region(id_muestra, x_obj, y_obj, tiles_encontrados):
    """Dibuja T1 / T2 / overlay para cada pipeline encontrado en la región."""
    fig, axes = plt.subplots(len(tiles_encontrados), 3, figsize=(15, 5 * len(tiles_encontrados)))
    axes = np.atleast_2d(axes)  # reemplaza el "if len==1: axes=[axes]" de antes

    for i, (nombre_pipeline, fila) in enumerate(tiles_encontrados.items()):
        t1 = cargar_rgb(fila["ruta_t1"])
        t2 = cargar_rgb(fila["ruta_t2"])
        overlay = cv2.addWeighted(t1, 0.5, t2, 0.5, 0)

        titulos = ["T1 Masson (ref)", f"T2 {nombre_pipeline}", f"Overlay {nombre_pipeline}"]
        for j, (img, titulo) in enumerate(zip([t1, t2, overlay], titulos)):
            axes[i][j].imshow(img)
            axes[i][j].set_title(titulo, fontsize=11)
            axes[i][j].axis("off")

        axes[i][0].text(
            -0.15, 0.5,
            f"{nombre_pipeline}\n{fila['nombre']}\ndist={fila['distancia_al_objetivo_px']:.0f}px",
            transform=axes[i][0].transAxes, rotation=90, va="center", ha="center", fontsize=9,
        )

    plt.suptitle(f"{id_muestra} - región ({x_obj:.0f}, {y_obj:.0f})", fontsize=14)
    plt.tight_layout()
    plt.savefig(
        os.path.join(CARPETA_SALIDA, f"comparacion_region_{id_muestra}_{int(x_obj)}_{int(y_obj)}.png"),
        dpi=150, bbox_inches="tight",
    )
    plt.show()


def _armar_tabla_metricas(id_muestra, tiles_encontrados):
    """Filtra df_metricas_todas a los tiles encontrados y arma la tabla pivot
    (filas = métrica, columnas = pipeline) con las 5 métricas."""
    condiciones = [
        (df_metricas_todas["pipeline"] == nombre_pipeline)
        & (df_metricas_todas["muestra"] == id_muestra)
        & (df_metricas_todas["nombre"] == fila["nombre"])
        for nombre_pipeline, fila in tiles_encontrados.items()
    ]
    tabla_region = df_metricas_todas[pd.concat(condiciones, axis=1).any(axis=1)]
    return tabla_region.set_index("pipeline")[METRICAS_PRINCIPALES].T


def _guardar_tabla_excel(tabla_pivot, id_muestra, x_obj, y_obj):
    ruta_excel = os.path.join(CARPETA_SALIDA, f"metricas_regiones_{id_muestra}.xlsx")
    nombre_hoja = f"x{int(x_obj)}_y{int(y_obj)}"[:31]
    modo = "a" if os.path.exists(ruta_excel) else "w"
    kwargs = {"if_sheet_exists": "replace"} if modo == "a" else {}
    with pd.ExcelWriter(ruta_excel, engine="openpyxl", mode=modo, **kwargs) as writer:
        tabla_pivot.round(4).to_excel(writer, sheet_name=nombre_hoja)
    print(f"Guardado en {ruta_excel} (hoja: {nombre_hoja})")


def comparar_region_3_pipelines(id_muestra, x_obj, y_obj, tol_px=TOLERANCIA_MATCH_PX, guardar_excel=True):
    """Compara los 3 pipelines en una región puntual: overlay visual + tabla de
    las 5 métricas principales del paper (PCC-TRE, NCC, SSIM enmascarado, Mask IoU,
    Folding ratio). Guarda la figura y, opcionalmente, la tabla en un Excel por muestra."""
    tiles_encontrados = _buscar_tiles_en_region(id_muestra, x_obj, y_obj, tol_px)
    if not tiles_encontrados:
        print("Ningún pipeline tiene un tile en esa región.")
        return None

    _graficar_overlay_region(id_muestra, x_obj, y_obj, tiles_encontrados)

    tabla_pivot = _armar_tabla_metricas(id_muestra, tiles_encontrados)
    print(f"\nMétricas para {id_muestra} en ({x_obj:.0f}, {y_obj:.0f}):")
    display(tabla_pivot.round(4))

    etiqueta_region = f"{id_muestra}_x{int(x_obj)}_y{int(y_obj)}"
    REGISTRO_REGIONES.append({"muestra": id_muestra, "region": etiqueta_region, "tabla": tabla_pivot})

    if guardar_excel:
        _guardar_tabla_excel(tabla_pivot, id_muestra, x_obj, y_obj)

    return tabla_pivot


def guardar_registro_regiones(id_muestra=None):
    """Consolida en un único CSV todas las tablas de región acumuladas en REGISTRO_REGIONES
    (todas, o solo las de una muestra si se pasa id_muestra)."""
    registros = (
        REGISTRO_REGIONES if id_muestra is None
        else [r for r in REGISTRO_REGIONES if r["muestra"] == id_muestra]
    )
    if not registros:
        print("No hay regiones registradas todavía.")
        return None

    filas = []
    for r in registros:
        tabla_larga = r["tabla"].T.reset_index().rename(columns={"index": "pipeline"})
        tabla_larga.insert(0, "region", r["region"])
        filas.append(tabla_larga)

    df_consolidado = pd.concat(filas, ignore_index=True)
    sufijo = id_muestra if id_muestra else "todas"
    ruta_csv = os.path.join(CARPETA_SALIDA, f"metricas_por_region_{sufijo}.csv")
    df_consolidado.to_csv(ruta_csv, index=False)
    print(f"Guardado en {ruta_csv}")
    return df_consolidado


tabla_region_ej1 = comparar_region_3_pipelines(id_muestra="Muestra1", x_obj=12288, y_obj=27024)
tabla_region_ej2 = comparar_region_3_pipelines(id_muestra="Muestra1", x_obj=19456, y_obj=28048)

df_metricas_regiones = guardar_registro_regiones(id_muestra="Muestra1")
df_metricas_regiones

# EMPAREJAR PIPELINES
Empareja tiles que existan en los 3 pipelines a la vez para la misma región, y para cada métrica corre Friedman seguido de las 3 comparaciones pareadas posibles con p-valores corregidos por Holm.

In [ ]:
from itertools import combinations
from scipy.stats import friedmanchisquare
from statsmodels.stats.multitest import multipletests

# UMBRALES DE LA LITERATURA
UMBRAL_NMI = 0.15          # UWarp (2025) - referencia, secundaria en este escenario monomodal
UMBRAL_IOU_APROBADO = 0.64 # URQA (2026) - "Aceptable" o mejor
UMBRAL_FOLDING_PCT = 1.5   # URQA (2026)


def emparejar_3_pipelines(id_muestra, tol_px=TOLERANCIA_MATCH_PX):
    df_a = df_sift_demons[df_sift_demons["muestra"] == id_muestra]
    df_b = df_dhr[df_dhr["muestra"] == id_muestra]
    df_c = df_valis[df_valis["muestra"] == id_muestra]
    if len(df_a) == 0 or len(df_b) == 0 or len(df_c) == 0:
        return pd.DataFrame()

    arbol_b = cKDTree(df_b[["x1_orig", "y1_orig"]].to_numpy(dtype=float))
    arbol_c = cKDTree(df_c[["x1_orig", "y1_orig"]].to_numpy(dtype=float))

    filas = []
    usados_b, usados_c = set(), set()
    for _, fila_a in df_a.iterrows():
        punto = np.array([fila_a["x1_orig"], fila_a["y1_orig"]])
        dist_b, idx_b = arbol_b.query(punto)
        dist_c, idx_c = arbol_c.query(punto)
        if dist_b <= tol_px and dist_c <= tol_px and idx_b not in usados_b and idx_c not in usados_c:
            usados_b.add(idx_b)
            usados_c.add(idx_c)
            filas.append({
                "muestra": id_muestra,
                "x1_orig": fila_a["x1_orig"], "y1_orig": fila_a["y1_orig"],
                "nombre_sift": fila_a["nombre"],
                "nombre_dhr": df_b.iloc[idx_b]["nombre"],
                "nombre_valis": df_c.iloc[idx_c]["nombre"],
            })
    return pd.DataFrame(filas)


ids_muestra = df_sift_demons["muestra"].unique()
df_triples_comunes = pd.concat([emparejar_3_pipelines(id_m) for id_m in ids_muestra], ignore_index=True)
print(f"Tiles con match en los 3 pipelines: {len(df_triples_comunes)}")
print(df_triples_comunes.groupby("muestra").size())


def construir_metricas_3vias(df_triples, metrica):
    idx = df_metricas_todas.set_index(["muestra", "pipeline", "nombre"])[metrica]
    valores = []
    for _, fila in df_triples.iterrows():
        try:
            v_sift = idx.loc[(fila["muestra"], "Sift_Demons", fila["nombre_sift"])]
            v_dhr = idx.loc[(fila["muestra"], "DeeperHistReg", fila["nombre_dhr"])]
            v_valis = idx.loc[(fila["muestra"], "VALIS", fila["nombre_valis"])]
            valores.append((v_sift, v_dhr, v_valis))
        except KeyError:
            continue
    return pd.DataFrame(valores, columns=["Sift_Demons", "DeeperHistReg", "VALIS"])


METRICAS_A_TESTEAR = [
    ("NCC", "mayor_mejor"), ("Mask_IoU", "mayor_mejor"),
    ("PCC_TRE_px", "menor_mejor"), ("Folding_ratio_pct", "menor_mejor"),
]

resultados_friedman = []
for metrica, direccion in METRICAS_A_TESTEAR:
    df_m = construir_metricas_3vias(df_triples_comunes, metrica)
    if len(df_m) < 5:
        print(f"{metrica}: muy pocos tiles emparejados ({len(df_m)}), se omite.")
        continue

    stat, p_omnibus = friedmanchisquare(df_m["Sift_Demons"], df_m["DeeperHistReg"], df_m["VALIS"])

    pares = list(combinations(df_m.columns, 2))
    p_valores = [wilcoxon(df_m[a], df_m[b])[1] for a, b in pares]
    _, p_ajustados, _, _ = multipletests(p_valores, method="holm")

    for (a, b), p_adj in zip(pares, p_ajustados):
        media_a, media_b = df_m[a].mean(), df_m[b].mean()
        mejor = a if (media_a > media_b) == (direccion == "mayor_mejor") else b
        resultados_friedman.append({
            "metrica": metrica, "n": len(df_m), "friedman_p": p_omnibus,
            "comparacion": f"{a} vs {b}", "p_ajustado_holm": p_adj,
            "significativo": p_adj < 0.05, "mejor": mejor,
        })

df_friedman_posthoc = pd.DataFrame(resultados_friedman)
df_friedman_posthoc.to_csv(os.path.join(CARPETA_SALIDA, "test_friedman_posthoc.csv"), index=False)
display(df_friedman_posthoc.round(4))

# Comparación gráfica por métrica.

Se muestran las 5 métricas principales del escenario monomodal (fila superior) y las 2 secundarias
(fila inferior).

In [ ]:
# GRAFICOS
METRICAS_A_GRAFICAR = [
    ("PCC_TRE_px", "PCC-TRE en píxeles (menor = mejor)", None, None),
    ("NCC", "NCC (mayor = mejor) — PRINCIPAL en monomodal", None, None),
    ("SSIM_enmascarado", "SSIM Enmascarado (mayor = mejor) — PRINCIPAL", None, None),
    ("Mask_IoU", "Mask IoU / Otsu (mayor = mejor)", UMBRAL_IOU_APROBADO, "umbral URQA (Aceptable)"),
    ("Folding_ratio_pct", "Folding ratio % (menor = mejor)", UMBRAL_FOLDING_PCT, "umbral URQA"),
]

# MODIFICACIÓN: Cambié n_cols=4 a n_cols=3
def graficar_boxplots_comparativos(df, metricas, nombre_archivo, hue_col=None, n_cols=3):
    """Dibuja un panel de boxplots (uno por métrica) comparando pipelines.
    Si hue_col se especifica (p.ej. 'muestra'), separa cada pipeline en
    sub-cajas por esa columna; si es None, un boxplot por pipeline.

    showfliers=False: con miles de tiles, los outliers individuales se
    superponen y forman un bloque sólido ilegible (cientos de círculos
    apilados). La caja + bigotes ya comunican el rango; los casos extremos
    puntuales se reportan mejor aparte, en una tabla de "peores tiles"."""
    plt.close("all")
    n_metricas = len(metricas)
    n_filas = int(np.ceil(n_metricas / n_cols))
    fig, axes = plt.subplots(n_filas, n_cols, figsize=(6 * n_cols, 5.5 * n_filas))
    axes = np.atleast_1d(axes).ravel()

    orden_pipelines = df["pipeline"].unique()
    orden_hue = df[hue_col].unique() if hue_col else [None]
    n_hue = len(orden_hue)

    for ax, (col, titulo, umbral, label_umbral) in zip(axes, metricas):
        sns.boxplot(
            data=df, x="pipeline", y=col,
            hue=(hue_col if hue_col else "pipeline"),
            ax=ax, palette="Set2", legend=bool(hue_col),
            showfliers=False,
        )

        cols_group = ["pipeline", hue_col] if hue_col else ["pipeline"]
        medianas = df.groupby(cols_group)[col].median()

        for i, pipe in enumerate(orden_pipelines):
            for j, h in enumerate(orden_hue):
                clave = (pipe, h) if hue_col else pipe
                if clave not in medianas.index:
                    continue
                valor = medianas.loc[clave]
                offset = (j - (n_hue - 1) / 2) * (0.8 / n_hue) if hue_col else 0
                ax.text(
                    i + offset, valor, f"{valor:.3f}",
                    ha="center", va="center", size="small", color="black", weight="semibold",
                    bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5),
                )

        if umbral is not None:
            ax.axhline(umbral, color="red", linestyle="--", linewidth=1.5, label=label_umbral)


        if col in ("Folding_ratio_pct", "PCC_TRE_px"):
            p99 = df[col].quantile(0.99)
            techo = max(p99 * 1.3, (umbral * 2 if umbral else p99 * 1.3))
            ax.set_ylim(-0.05 * techo, techo)

        ax.set_title(titulo, fontsize=11, pad=10)
        ax.set_xlabel("")
        ax.set_ylabel(col, labelpad=8)
        ax.tick_params(axis="x", rotation=15)
        if ax.get_legend() is not None:
            ax.legend(fontsize=8, loc="best")

    for ax in axes[n_metricas:]:
        ax.axis("off")

    fig.tight_layout()
    plt.savefig(os.path.join(CARPETA_SALIDA, nombre_archivo), dpi=150, bbox_inches="tight")
    plt.show()


# Vista 1: comparación global por pipeline (sin separar por muestra) 
graficar_boxplots_comparativos(
    df_metricas_todas, METRICAS_A_GRAFICAR,
    nombre_archivo="boxplots_comparativos_pipelines.png",
    hue_col=None,
)

# Vista 2: la misma comparación, separada por muestra 
graficar_boxplots_comparativos(
    df_metricas_todas, METRICAS_A_GRAFICAR,
    nombre_archivo="boxplots_comparativos_por_muestra.png",
    hue_col="muestra",
)

# Evaluación Final: Set de 5 Métricas y Análisis Estadístico

Este bloque de código ejecuta el pipeline de validación estadística completo:

*   **Análisis de Correlación (Spearman):** Se calcula una matriz de correlación entre las 5 métricas para demostrar su independencia. Esto justifica metodológicamente la inclusión de todas ellas, evidenciando que miden características distintas del registro (error espacial, similitud de intensidad, superposición macroscópica y plausibilidad física) sin ser redundantes.
*   **Emparejamiento Espacial Estricto:** Se construye un nuevo conjunto de datos (`df_pareado_v2`) que cruza los resultados de los 3 pipelines evaluando *exactamente la misma ubicación física* (coordenadas de origen). Esto es un requisito indispensable para un análisis comparativo justo.
*   **Test de Wilcoxon Signed-Rank:** Se aplican pruebas no paramétricas pareadas para cada combinación de pipelines. Esto permite afirmar con significancia estadística (p < 0.05) si un método supera realmente a otro.
*   **Tasa de Victorias (Win Rate):** Se calcula el porcentaje de veces que cada pipeline obtiene el primer puesto absoluto por métrica, ofreciendo una métrica intuitiva de rendimiento global.
*   **Aprobación según Umbrales de la Literatura:** Se genera la tabla resumen final (medias y medianas) que incluye la tasa de éxito clínico/técnico frente a estándares publicados, como el umbral URQA para Mask IoU (≥ 0.64) y tolerancia a deformaciones no plausibles (Folding < 1.5%).

In [ ]:
assert "df_metricas_todas" in globals(), "Falta 'df_metricas_todas'."
assert "df_pares_comunes" in globals(), "Falta 'df_pares_comunes' — correr antes EMPAREJAR PIPELINES."
assert "CARPETA_SALIDA" in globals(), "Falta 'CARPETA_SALIDA'."

# UMBRALES DE LA LITERATURA
UMBRAL_NMI = 0.15          # UWarp (2025) - referencia, secundaria en este escenario monomodal
UMBRAL_IOU_APROBADO = 0.64 # URQA (2026) - "Aceptable" o mejor
UMBRAL_FOLDING_PCT = 1.5   # URQA (2026)

from itertools import combinations
from scipy.stats import wilcoxon, spearmanr

METRICAS_PRINCIPALES_V2 = ["PCC_TRE_px", "NCC", "SSIM_enmascarado", "Mask_IoU", "Folding_ratio_pct"]
DIRECCION_V2 = {
    "PCC_TRE_px": "menor_mejor", "NCC": "mayor_mejor", "SSIM_enmascarado": "mayor_mejor",
    "Mask_IoU": "mayor_mejor", "Folding_ratio_pct": "menor_mejor",
}

# Correlación del set de 5 métricas principales (subset, confirma no-redundancia)
def orientar(df, metricas, direcciones):
    d = df[metricas].copy()
    for m in metricas:
        if direcciones[m] == "menor_mejor":
            d[m] = -d[m]
    return d

df_orientado_v2 = orientar(df_metricas_todas, METRICAS_PRINCIPALES_V2, DIRECCION_V2)
matriz_corr_v2 = df_orientado_v2.corr(method="spearman")

plt.close("all")
fig, ax = plt.subplots(figsize=(9, 7.5))  
sns.heatmap(
    matriz_corr_v2, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    vmin=-1, vmax=1, square=True, linewidths=0.6, linecolor="white",
    cbar_kws={"label": "Spearman r\n(orientado a 'mayor=mejor')", "shrink": 0.75},
    annot_kws={"fontsize": 11},
    ax=ax,
)
ax.set_title(
    "Matriz de correlación\n"
    "(PCC-TRE, NCC, SSIM enmascarado, Mask IoU, Folding ratio)",
    fontsize=12, pad=18,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)


fig.subplots_adjust(left=0.28, right=0.93, top=0.85, bottom=0.30)

plt.savefig(os.path.join(CARPETA_SALIDA, "matriz_correlacion_5_metricas.png"), dpi=150, bbox_inches="tight")
plt.show()
matriz_corr_v2.round(3).to_csv(os.path.join(CARPETA_SALIDA, "matriz_correlacion_5_metricas.csv"))


def construir_metricas_pareadas(df_pares_comunes, dfs_metricas):
    idx = {nom: df.set_index("nombre") for nom, df in dfs_metricas.items()}
    filas = []
    for _, par in df_pares_comunes.iterrows():
        fila = {"muestra": par["muestra"], "x1_orig": par["x1_orig"], "y1_orig": par["y1_orig"]}
        datos_pipeline, ok = {}, True
        for nom in idx:
            nombre_tile = par[f"nombre_{nom}"]
            if nombre_tile not in idx[nom].index:
                ok = False
                break
            fila_m = idx[nom].loc[nombre_tile]
            if isinstance(fila_m, pd.DataFrame):
                fila_m = fila_m.iloc[0]
            datos_pipeline[nom] = fila_m
        if not ok:
            continue
        for nom, fila_m in datos_pipeline.items():
            fila[f"NCC_{nom}"] = fila_m["NCC"]
            fila[f"SSIMmask_{nom}"] = fila_m["SSIM_enmascarado"]
            fila[f"IoU_{nom}"] = fila_m["Mask_IoU"]
            fila[f"TRE_{nom}"] = fila_m["PCC_TRE_px"]
            fila[f"Folding_{nom}"] = fila_m["Folding_ratio_pct"]
        filas.append(fila)
    return pd.DataFrame(filas)

df_pareado_v2 = construir_metricas_pareadas(
    df_pares_comunes,
    {"Sift_Demons": df_metricas_sift_demons, "DeeperHistReg": df_metricas_dhr, "VALIS": df_metricas_valis},
)
print(f"\nTiles pareados: {len(df_pareado_v2)}")


# Wilcoxon pareado
pares_metricas = [
    ("NCC", "mayor_mejor"), ("SSIMmask", "mayor_mejor"), ("IoU", "mayor_mejor"),
    ("TRE", "menor_mejor"), ("Folding", "menor_mejor"),
]
resultados_wilcoxon_v2 = []
if len(df_pareado_v2) >= 5:
    for base, direccion in pares_metricas:
        for pip_a, pip_b in combinations(PIPELINES, 2):
            col_a, col_b = f"{base}_{pip_a}", f"{base}_{pip_b}"
            try:
                stat, p_value = wilcoxon(df_pareado_v2[col_a], df_pareado_v2[col_b])
            except ValueError:
                stat, p_value = np.nan, np.nan
            media_a, media_b = df_pareado_v2[col_a].mean(), df_pareado_v2[col_b].mean()
            ganador = (pip_a if media_a > media_b else pip_b) if direccion == "mayor_mejor" \
                      else (pip_a if media_a < media_b else pip_b)
            resultados_wilcoxon_v2.append({
                "metrica": base, "comparacion": f"{pip_a} vs {pip_b}", "n_pares": len(df_pareado_v2),
                f"media_{pip_a}": media_a, f"media_{pip_b}": media_b,
                "p_value": p_value, "significativo_p<0.05": (p_value < 0.05) if not np.isnan(p_value) else None,
                "mejor_segun_media": ganador,
            })
    df_resultados_wilcoxon_v2 = pd.DataFrame(resultados_wilcoxon_v2)
    print("\nTEST DE WILCOXON")
    display(df_resultados_wilcoxon_v2.round(4))
    df_resultados_wilcoxon_v2.round(4).to_csv(os.path.join(CARPETA_SALIDA, "test_wilcoxon_5_metricas.csv"), index=False)


# Win rate por tile 
filas_win = []
for base, direccion in pares_metricas:
    cols = [f"{base}_{pip}" for pip in PIPELINES]
    cols_presentes = [c for c in cols if c in df_pareado_v2.columns]
    if len(cols_presentes) < 2:
        continue
    ganadores_col = df_pareado_v2[cols_presentes].apply(
        lambda fila: (fila.idxmax() if direccion == "mayor_mejor" else fila.idxmin()), axis=1
    )
    ganadores = ganadores_col.apply(lambda c: c[len(f"{base}_"):])
    conteo, n_total = ganadores.value_counts(), len(ganadores)
    fila_resultado = {"metrica": base, "n_tiles": n_total}
    for pip in PIPELINES:
        fila_resultado[f"win_%_{pip}"] = round(100 * conteo.get(pip, 0) / n_total, 1)
    filas_win.append(fila_resultado)

df_win_rate = pd.DataFrame(filas_win)
print("\nWIN RATE")
display(df_win_rate)
df_win_rate.to_csv(os.path.join(CARPETA_SALIDA, "tabla_win_rate_5_metricas.csv"), index=False)

cols_win_v2 = [f"win_%_{pip}" for pip in PIPELINES]
plt.close("all")
fig, ax = plt.subplots(figsize=(8.5, 5.5))
df_plot_v2 = df_win_rate.set_index("metrica")[cols_win_v2]
df_plot_v2.columns = [c.replace("win_%_", "") for c in df_plot_v2.columns]
df_plot_v2.plot(kind="bar", stacked=True, ax=ax, colormap="Set2", width=0.65)
ax.set_ylabel("% de tiles ganados")
ax.set_xlabel("")
ax.set_title(f"Win rate por métrica (n={len(df_pareado_v2)})", fontsize=12, pad=16)
ax.legend(title="Pipeline", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f%%", label_type="center", fontsize=9)
fig.subplots_adjust(right=0.8)
plt.savefig(os.path.join(CARPETA_SALIDA, "win_rate_5_metricas.png"), dpi=150, bbox_inches="tight")
plt.show()


# Tabla resumen final (media/mediana/std + % aprobado)
def resumen_pipeline(df):
    resumen = df[METRICAS_PRINCIPALES_V2].agg(["mean", "median", "std"]).T
    resumen.columns = ["media", "mediana", "std"]
    resumen.loc["pct_aprobado_IoU_geq_0.64", "media"] = 100 * (df["Mask_IoU"] >= UMBRAL_IOU_APROBADO).mean()
    resumen.loc["pct_aprobado_folding_lt_1.5pct", "media"] = 100 * (df["Folding_ratio_pct"] < UMBRAL_FOLDING_PCT).mean()
    return resumen

tablas_v2 = {}
for nombre_pipeline, df_m in [("Sift_Demons", df_metricas_sift_demons),
                               ("DeeperHistReg", df_metricas_dhr),
                               ("VALIS", df_metricas_valis)]:
    if len(df_m):
        tablas_v2[nombre_pipeline] = resumen_pipeline(df_m)

tabla_comparativa = pd.concat(
    {nombre: tabla[["media", "mediana", "std"]] for nombre, tabla in tablas_v2.items()}, axis=1
)
print("\nTABLA COMPARATIVA")
display(tabla_comparativa.round(4))
tabla_comparativa.round(4).to_csv(os.path.join(CARPETA_SALIDA, "tabla_comparativa_5_metricas.csv"))

# Guardado de resultados

Comprime la carpeta `CARPETA_SALIDA` (todas las tablas CSV y figuras PNG generadas por las celdas de arriba) en un único `.zip` para descargar.

In [ ]:
# GUARDADO DE TILES EN CARPETA .ZIP
import shutil
import os
from IPython.display import FileLink, display


# Compresión y exportación del dataset
os.chdir('/kaggle/working')

carpeta_tiles = '/kaggle/working/comparativa_pipelines'
nombre_zip_base = '/kaggle/working/comparacion_pipelines'

print(f"\nComprimiendo la carpeta: {carpeta_tiles}...")

# Comprimir la carpeta entera en formato ZIP
shutil.make_archive(nombre_zip_base, 'zip', carpeta_tiles)

archivo_generado = f"{nombre_zip_base}.zip"
peso_mb = os.path.getsize(archivo_generado) / (1024 * 1024)


print(f"\nCompresión finalizada!")
print(f"Archivo: {archivo_generado}")
print(f"Peso aproximado: {peso_mb:.2f} MB")

# Generar un link para descargar 
display(FileLink('comparacion_pipelines.zip'))